# Step 1 — Load BERT Directly

Instead of using the high-level `pipeline()` API, we will load
the BERT Question Answering model directly.

We will use two components:

### AutoTokenizer

Converts human language into tokens/numbers that **BERT**  can understand.

### AutoModelForQuestionAnswering

Loads a pretrained BERT model designed for Question Answering.

Architecture:

Human Question
      ⬇
Tokenizer
      ↓
Numbers / Tokens
      ↓
BERT QA Model
      ↓
Start + End Positions
      ↓
Answer

**Step 2**

In [1]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

model_name = "deepset/bert-base-cased-squad2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForQuestionAnswering.from_pretrained(model_name)

print("Tokenizer loaded successfully!")
print("BERT Question Answering model loaded successfully!")

config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/152 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: deepset/bert-base-cased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer loaded successfully!
BERT Question Answering model loaded successfully!


# Step 3 — Create Our Airport Knowledge

Our model is pretrained, but we want to use it for a specific domain.

Our domain:

## ✈️ Airport Information

We will provide the model with a paragraph containing
airport-specific information.

This is our domain-specific context.

In [ ]:
context = """
Mumbai International Airport operates 24 hours a day.

Terminal 1 handles domestic flights and Terminal 2 handles
international flights and selected domestic flights.

Domestic passengers should arrive at least two hours before
their scheduled departure.

International passengers should arrive at least three hours
before departure.

The airport provides free Wi-Fi to passengers.

Passengers can find restaurants, cafes, lounges and shopping
facilities inside the terminals.

Airport check-in counters generally close 45 minutes before
domestic flights and 60 minutes before international flights.

The airport provides wheelchair assistance for passengers
who request the service.

Passengers can use taxis, buses and private vehicles to reach
the airport.

The airport has dedicated parking areas for cars and
two-wheelers.
"""

print(context)


Mumbai International Airport operates 24 hours a day.

Terminal 1 handles domestic flights and Terminal 2 handles
international flights and selected domestic flights.

Domestic passengers should arrive at least two hours before
their scheduled departure.

International passengers should arrive at least three hours
before departure.

The airport provides free Wi-Fi to passengers.

Passengers can find restaurants, cafes, lounges and shopping
facilities inside the terminals.

Airport check-in counters generally close 45 minutes before
domestic flights and 60 minutes before international flights.

The airport provides wheelchair assistance for passengers
who request the service.

Passengers can use taxis, buses and private vehicles to reach
the airport.

The airport has dedicated parking areas for cars and
two-wheelers.



# Step 4 — Create a Question

Let's ask our AI:

> How early should domestic passengers arrive?

The model will receive:

Question
+
Context

and determine which part of the context answers the question.

In [ ]:
question = "How early should domestic passengers arrive?"

print("Question:", question)

Question: How early should domestic passengers arrive?


# Step 5 — Tokenization

Computers do not directly understand:

"How early should domestic passengers arrive?"

The tokenizer converts the question and context into
tokens and numerical representations.

Conceptually:

Text
 ↓
Tokenizer
 ↓
Tokens
 ↓
Token IDs
 ↓
BERT

In [ ]:
inputs = tokenizer(
    question,
    context,
    return_tensors="pt"
)

print(inputs.keys())

KeysView({'input_ids': tensor([[  101,  1731,  1346,  1431,  4500,  4861,  6657,   136,   102,  8601,
          1570,  3369,  5049,  1572,  2005,   170,  1285,   119, 11844,   122,
         17180,  4500,  7306,  1105, 11844,   123, 17180,  1835,  7306,  1105,
          2700,  4500,  7306,   119, 21057,  4861,  1431,  6657,  1120,  1655,
          1160,  2005,  1196,  1147,  4533,  6267,   119,  1570,  4861,  1431,
          6657,  1120,  1655,  1210,  2005,  1196,  6267,   119,  1109,  3871,
          2790,  1714,   160,  1182,   118, 17355,  1106,  4861,   119, 20252,
          1116,  1169,  1525,  7724,   117, 17287,  1116,   117, 15238,  1116,
          1105,  6001,  3380,  1656,  1103, 20618,   119,  3369,  4031,   118,
          1107, 27633,  2412,  1601,  2532,  1904,  1196,  4500,  7306,  1105,
          2539,  1904,  1196,  1835,  7306,   119,  1109,  3871,  2790, 14980,
          5052,  1111,  4861,  1150,  4566,  1103,  1555,   119, 20252,  1116,
          1169,  1329, 11736,

# Step 6 — Send the Input to BERT

Now we send our tokenized question and context into
the pretrained BERT Question Answering model.

Important:

We are performing **inference**.

We are NOT training the model.

In [ ]:
import torch
from transformers import AutoModelForQuestionAnswering

model_name = "deepset/bert-base-cased-squad2"

model = AutoModelForQuestionAnswering.from_pretrained(model_name)

with torch.no_grad():
    outputs = model(**inputs)
print(outputs.keys())
print("Model inference completed!")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: deepset/bert-base-cased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


odict_keys(['start_logits', 'end_logits'])
Model inference completed!


# Step 7 — Understand BERT's Output

The Question Answering model gives us two important outputs:

### Start Logits

Scores indicating where the answer may begin.

### End Logits

Scores indicating where the answer may end.

Conceptually:

Context:

"Domestic passengers should arrive at least two hours
before their scheduled departure."

                         ↓

Start Position ──────────┐

                         ↓
        "at least two hours"
                         ↑
End Position ────────────┘

In [ ]:
start_logits = outputs.start_logits
end_logits = outputs.end_logits

print("Start logits shape:", start_logits.shape)
print("End logits shape:", end_logits.shape)

Start logits shape: torch.Size([1, 149])
End logits shape: torch.Size([1, 149])


# Step 8 — Find the Most Likely Start and End Positions

We use the highest-scoring start and end positions.

This gives us the section of text that BERT believes
contains the answer.

In [ ]:
start_position = torch.argmax(start_logits, dim=1).item()
end_position = torch.argmax(end_logits, dim=1).item()

print("Start position:", start_position)
print("End position:", end_position)

Start position: 40
End position: 41


# Step 9 — Convert the Answer Tokens Back to Text

We now have:

Start position
+
End position

We need to convert those tokens back into human-readable text.

In [ ]:
answer_tokens = inputs["input_ids"][0][start_position:end_position + 1]

answer = tokenizer.decode(
    answer_tokens,
    skip_special_tokens=True
)

print("Question:", question)
print("Answer:", answer)

Question: How early should domestic passengers arrive?
Answer: two hours


# 🧠 What Just Happened?

Our complete process was:

User Question

      ↓
Tokenizer

      ↓
Token IDs

      ↓
BERT

      ↓
Start Logits + End Logits

      ↓
Start Position + End Position

      ↓
Answer Tokens

      ↓
Tokenizer Decode

      ↓
Human-readable Answer


This is the basic mechanism behind extractive
Question Answering.

# Step 10 — Create a Reusable Question Answering Function

We will now turn everything we learned into one reusable function.

The function will:

1. Receive a question.
2. Tokenize question + context.
3. Run BERT.
4. Find the best answer span.
5. Convert the answer back to text.
6. Return the answer.

In [ ]:
def answer_question(question, context):

    # Tokenize question and context
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt"
    )

    # Run model
    with torch.no_grad():
        outputs = model(**inputs)

    # Get start and end logits
    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]

    # Get most likely start and end positions
    start_position = torch.argmax(start_logits).item()
    end_position = torch.argmax(end_logits).item()

    # Make sure end comes after start
    if end_position < start_position:
        return "Unable to find a valid answer."

    # Extract answer tokens
    answer_tokens = inputs["input_ids"][0][
        start_position:end_position + 1
    ]

    # Convert tokens back to text
    answer = tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    )

    return answer

In [ ]:
answer = answer_question(
    "How early should international passengers arrive?",
    context
)

print("Answer:", answer)

Answer: three hours


In [ ]:
answer = answer_question(
    "Does the airport provide Wi-Fi?",
    context
)

print("Answer:", answer)

Answer: free


In [ ]:
questions = [
    "How early should domestic passengers arrive?",
    "How early should international passengers arrive?",
    "Does the airport provide Wi-Fi?",
    "Which terminal handles international flights?",
    "Is wheelchair assistance available?"
]

for question in questions:

    answer = answer_question(
        question,
        context
    )

    print("Question:", question)
    print("Answer:", answer)
    print("-" * 60)

Question: How early should domestic passengers arrive?
Answer: two hours
------------------------------------------------------------
Question: How early should international passengers arrive?
Answer: three hours
------------------------------------------------------------
Question: Does the airport provide Wi-Fi?
Answer: free
------------------------------------------------------------
Question: Which terminal handles international flights?
Answer: Terminal 2
------------------------------------------------------------
Question: Is wheelchair assistance available?
Answer: The airport provides wheelchair assistance for passengers who request the service
------------------------------------------------------------


In [ ]:
answer = answer_question(
    "alkdjalksjdlaj?",
    context
)

print("Answer:", answer)

Answer: 
